# China Trade Flow — Sankey Visualization

Left stacked bar: all countries by total trade (2024 BACI, metric tons).  
Sankey flows: China → top 5 export partners (upper), top 5 import partners → China (lower).  
Flow widths are true-scale — they show how much heavier China's imports are vs exports by weight.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# ── Load total trade matrix ───────────────────────────────────────────────
matrix_path = '../../data/all_trade_matrices/weight_trade_matrix_all_transport_modes_TOTAL.csv'
df = pd.read_csv(matrix_path, index_col=0)
df = df.drop('World', axis=0, errors='ignore').drop('World', axis=1, errors='ignore')

CHINA = 'China'
# Exclude HK/Macao/Taiwan — they are separate BACI entities but part of China proper
CHINA_EXCL = {'China', 'China, Hong Kong SAR', 'China, Macao SAR', 'China, Taiwan Province of'}

china_ex = df.loc[CHINA].drop([c for c in CHINA_EXCL if c in df.columns], errors='ignore')
china_im = df[CHINA].drop([c for c in CHINA_EXCL if c in df.index],   errors='ignore')

top5_ex = china_ex.nlargest(5)
top5_im = china_im.nlargest(5)

print('Top 5 export partners (MT):')
for c, v in top5_ex.items(): print(f'  {c}: {v/1e6:.1f}M MT')
print('Top 5 import partners (MT):')
for c, v in top5_im.items(): print(f'  {c}: {v/1e6:.1f}M MT')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────

def draw_flow(ax, x0, yb0, yt0, x1, yb1, yt1, color='#aaaaaa', alpha=0.45):
    """Fill a Sankey ribbon using cubic Bezier with flat tangents."""
    t  = np.linspace(0, 1, 300)
    mx = (x0 + x1) / 2
    # x follows S-curve; y edges follow separate flat-tangent cubics
    bx = (1-t)**3*x0 + 3*(1-t)**2*t*mx + 3*(1-t)*t**2*mx + t**3*x1
    by = (1-t)**3*yb0 + 3*(1-t)**2*t*yb0 + 3*(1-t)*t**2*yb1 + t**3*yb1
    ty = (1-t)**3*yt0 + 3*(1-t)**2*t*yt0 + 3*(1-t)*t**2*yt1 + t**3*yt1
    xs = np.concatenate([bx, bx[::-1]])
    ys = np.concatenate([by, ty[::-1]])
    ax.fill(xs, ys, color=color, alpha=alpha, lw=0, zorder=1)


def layout_nodes(series, y_start, y_end, gap=0.012):
    """Lay out partner nodes proportionally between y_start and y_end."""
    total    = series.sum()
    n        = len(series)
    avail    = (y_end - y_start) - gap * (n - 1)
    pos, y   = {}, y_start
    for c, v in series.items():
        h       = avail * v / total
        pos[c]  = (y, y + h)
        y      += h + gap
    return pos


def fmt_mt(v):
    """Short label: e.g. '928M MT' or '51M MT'."""
    if v >= 1e9: return f'{v/1e9:.2f}B MT'
    return f'{v/1e6:.0f}M MT'

In [ ]:
# ── Layout constants ──────────────────────────────────────────────────────
LX, LW = 0.04, 0.07    # left bar
RX, RW = 0.72, 0.055   # right partner bars

# ── Left stacked bar: all countries sorted by total trade desc ────────────
total_exports = df.sum(axis=1)
total_imports = df.sum(axis=0)
total_trade   = total_exports + total_imports
total_sum     = total_trade.sum()

countries_sorted = total_trade.sort_values(ascending=False).index.tolist()

left_y = {}
y = 0.0
for c in countries_sorted:
    h        = float(total_trade[c]) / total_sum
    left_y[c] = (y, y + h)
    y        += h

# ── China's bar segment ───────────────────────────────────────────────────
cy0, cy1 = left_y[CHINA]
ch       = cy1 - cy0

# Allocate China's segment: flows scaled by true weight
shown_ex = top5_ex.sum()
shown_im = top5_im.sum()
shown_total = shown_ex + shown_im

# Bottom portion → import flows; top portion → export flows
im_band_h = ch * (shown_im / shown_total)
ex_band_h = ch * (shown_ex / shown_total)

china_im_slots = {}
y = cy0
for c, v in top5_im.items():
    h = ch * (v / shown_total)
    china_im_slots[c] = (y, y + h)
    y += h

china_ex_slots = {}
y = cy0 + im_band_h
for c, v in top5_ex.items():
    h = ch * (v / shown_total)
    china_ex_slots[c] = (y, y + h)
    y += h

# ── Right-side partner nodes ──────────────────────────────────────────────
# Export partners: upper half of figure; import partners: lower half
ex_node_y = layout_nodes(top5_ex, 0.53, 0.95)
im_node_y = layout_nodes(top5_im, 0.05, 0.47)

print(f'China bar: [{cy0:.3f}, {cy1:.3f}]  height={ch:.4f}')
print(f'Import band: {im_band_h:.4f}  Export band: {ex_band_h:.4f}')

In [ ]:
# ── Draw ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 11))
ax.set_xlim(-0.12, 1.0)
ax.set_ylim(0, 1)
ax.axis('off')
fig.patch.set_facecolor('white')

# --- Left stacked bar ---
for i, c in enumerate(countries_sorted):
    y0, y1 = left_y[c]
    if c == CHINA:
        color = '#4a4a4a'
    elif i % 2 == 0:
        color = '#cccccc'
    else:
        color = '#b8b8b8'
    ax.add_patch(patches.Rectangle(
        (LX, y0), LW, y1 - y0,
        fc=color, ec='white', lw=0.15, zorder=2
    ))

# China arrow + label
china_mid = (cy0 + cy1) / 2
ax.annotate(
    'China', xy=(LX, china_mid), xytext=(LX - 0.045, china_mid),
    fontsize=11, fontweight='bold', color='#1a1a1a', ha='right', va='center',
    arrowprops=dict(arrowstyle='->', color='#333333', lw=1.2)
)
ax.text(LX + LW / 2, 1.015, 'All countries\n(total trade)', ha='center',
        va='bottom', fontsize=8, color='#555555')

# --- Sankey flows ---
# Export flows (darker grey)
for c in top5_ex.index:
    cy_b, cy_t = china_ex_slots[c]
    ry_b, ry_t = ex_node_y[c]
    draw_flow(ax, LX + LW, cy_b, cy_t, RX, ry_b, ry_t, color='#777777', alpha=0.50)

# Import flows (lighter grey)
for c in top5_im.index:
    cy_b, cy_t = china_im_slots[c]
    ry_b, ry_t = im_node_y[c]
    draw_flow(ax, LX + LW, cy_b, cy_t, RX, ry_b, ry_t, color='#aaaaaa', alpha=0.45)

# --- Right partner bars + labels ---
def draw_partner_bars(node_dict, value_series, label_suffix=''):
    for c, (y0, y1) in node_dict.items():
        ax.add_patch(patches.Rectangle(
            (RX, y0), RW, y1 - y0,
            fc='#999999', ec='white', lw=0.4, zorder=2
        ))
        label = f'{c}\n{fmt_mt(value_series[c])}'
        ax.text(RX + RW + 0.012, (y0 + y1) / 2, label,
                ha='left', va='center', fontsize=9, color='#1a1a1a',
                linespacing=1.4)

draw_partner_bars(ex_node_y, top5_ex)
draw_partner_bars(im_node_y, top5_im)

# --- Section labels between left bar and flows ---
mid_x = (LX + LW + RX) / 2
ax.text(mid_x, 0.975, 'Exports  →', ha='center', va='top',
        fontsize=9.5, color='#444444', style='italic')
ax.text(mid_x, 0.495, '←  Imports', ha='center', va='top',
        fontsize=9.5, color='#666666', style='italic')

# Thin horizontal divider in China's bar to separate export/import allocation
divider_y = cy0 + im_band_h
ax.plot([LX, LX + LW], [divider_y, divider_y],
        color='white', lw=0.8, zorder=3)

# --- Title ---
ax.set_title(
    "China's Top 5 Export & Import Partners — by Weight (Metric Tons, 2024 BACI)",
    fontsize=13, pad=14, color='#1a1a1a'
)

# --- Subtle footnote ---
ax.text(0.5, -0.015,
        'Flow widths are true-scale. Imports dominate by weight: China imports bulk commodities (ore, coal, soy), exports manufactured goods.',
        ha='center', va='top', transform=ax.transAxes,
        fontsize=7.5, color='#777777', style='italic')

plt.tight_layout()
out_path = 'visualization_outputs/china_trade_sankey.png'
import os; os.makedirs('visualization_outputs', exist_ok=True)
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved to {out_path}')
plt.show()